# Auto Loan Portfolio: Default & Risk Analysis
A concise BI analysis focused on portfolio segmentation and collections prioritization.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

## 1. Load and review the portfolio

In [ ]:
data_path = Path('../data/raw/train.csv')
df = pd.read_csv(data_path)
df.columns = df.columns.str.lower()
print(f'Rows: {len(df):,} | Columns: {df.shape[1]}')
df.head()

In [ ]:
pd.Series({
    'Total loans': len(df),
    'Total disbursed': df.disbursed_amount.sum(),
    'Average loan': df.disbursed_amount.mean(),
    'Average LTV': df.ltv.mean(),
    'Default rate': df.loan_default.mean(),
})

## 2. Segment by LTV

In [ ]:
df['ltv_band'] = pd.cut(df.ltv, [0, 60, 70, 80, 90, float('inf')], labels=['<=60%', '60-70%', '70-80%', '80-90%', '>90%'], include_lowest=True)
ltv_summary = df.groupby('ltv_band', observed=True).agg(loans=('uniqueid', 'count'), default_rate=('loan_default', 'mean')).reset_index()
ltv_summary

In [ ]:
ax = sns.barplot(data=ltv_summary, x='ltv_band', y='default_rate', color='#2F75B5')
ax.yaxis.set_major_formatter(lambda x, pos: f'{x:.0%}')
ax.set(title='Default Rate by LTV Band', xlabel='LTV band', ylabel='Default rate');

## 3. Recent delinquency

In [ ]:
df['prior_delinquency'] = (df.delinquent_accts_in_last_six_months > 0).map({True: 'Yes', False: 'No'})
df.groupby('prior_delinquency').agg(loans=('uniqueid', 'count'), default_rate=('loan_default', 'mean'))

## Conclusion
The 80-90% LTV segment, recent-delinquency accounts and mid-to-low bureau-score segments deserve earlier monitoring. This is a prioritization analysis, not a complete collections performance model: monthly DPD, cure, roll-rate, contact and recovery data are unavailable.